# 05 — Exportação para TFLite (float32 e int8)

Converte os três modelos finais (`aug+ft` do notebook 04) para TFLite, que é o formato usado no app Android:

| Arquivo | Descrição |
|---|---|
| `<run>_fp32.tflite` | pesos em float32 (referência) |
| `<run>_int8.tflite` | quantização inteira de 8 bits (pesos e ativações), entrada/saída em float32 |
| `labels.txt` | as 38 classes, na ordem das saídas do modelo |
| `model_info.json` | formato de entrada, classes e limiar sugerido para o app |

Todos os modelos recebem **imagem RGB 128×128 em float32 com pixels 0..255** (a normalização está dentro do modelo)
e devolvem **38 probabilidades** (softmax).

Também mede, para cada arquivo: tamanho, acurácia (PlantVillage e PlantDoc) e latência na CPU do Colab.
**Atenção:** na CPU x86 do Colab o int8 pode até sair mais lento que o float32; o ganho de velocidade
do int8 aparece em processadores ARM (celular). A latência que vai para o artigo é a medida no app.

Saídas em `MyDrive/leafscan/results/tflite/` e tabela em `results/tables/tab_tflite.csv`/`.tex`.
Tempo esperado: ~10–15 min (a avaliação do int8 na CPU é a parte mais lenta).

## 1. Configuração e dados

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import accuracy_score, f1_score

DRIVE_DIR = '/content/drive/MyDrive/leafscan'
CACHE_DIR = f'{DRIVE_DIR}/cache'
RUNS_DIR  = f'{DRIVE_DIR}/runs'
OUT_DIR   = f'{DRIVE_DIR}/results/tflite'
TAB_DIR   = f'{DRIVE_DIR}/results/tables'
LOCAL     = '/content/cache'
SEED      = 42
RUNS      = ['cnn_base', 'mnv3s', 'mnv2_050']
NAMES     = {'cnn_base': 'CNN própria', 'mnv3s': 'MobileNetV3-Small', 'mnv2_050': 'MobileNetV2 (α=0.5)'}
N_PV_EVAL = 2000          # amostra do teste do PlantVillage para avaliar os .tflite (a CPU é lenta)
N_REP     = 300           # imagens de calibração da quantização int8
THRESHOLD = 0.7           # limiar de confiança sugerido para o app (ver notebook 04)

for d in (OUT_DIR, TAB_DIR, LOCAL):
    os.makedirs(d, exist_ok=True)
for r in RUNS:
    assert os.path.exists(f'{RUNS_DIR}/{r}_aug_ft/best.keras'), f'Falta {r}_aug_ft: rode o notebook 04'

for name in ('pv_x', 'pv_y', 'pv_split', 'pd_x', 'pd_y', 'pd_split'):
    src, dst = f'{CACHE_DIR}/{name}.npy', f'{LOCAL}/{name}.npy'
    for tentativa in range(3):
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            break
        shutil.copy(src, dst + '.tmp'); os.replace(dst + '.tmp', dst)
    assert os.path.getsize(dst) == os.path.getsize(src), f'Cópia de {name} incompleta'

meta = json.load(open(f'{CACHE_DIR}/meta.json'))
PV_CLASSES = meta['pv_classes']; NUM_CLASSES = len(PV_CLASSES)
PD_IDX = np.array(meta['pd_pv_indices']); IMG = meta['img_size']

pv_x = np.load(f'{LOCAL}/pv_x.npy', mmap_mode='r'); pv_y = np.load(f'{LOCAL}/pv_y.npy').astype(int)
pv_split = np.load(f'{LOCAL}/pv_split.npy')
pd_x = np.load(f'{LOCAL}/pd_x.npy'); pd_y = np.load(f'{LOCAL}/pd_y.npy').astype(int)
pd_split = np.load(f'{LOCAL}/pd_split.npy')

rng = np.random.default_rng(SEED)
i_te = np.sort(rng.choice(np.where(pv_split == 2)[0], N_PV_EVAL, replace=False))
x_te, y_te = np.asarray(pv_x[i_te]), pv_y[i_te]
x_pd, y_pd = pd_x[pd_split == 2], pd_y[pd_split == 2]

# calibração do int8: metade PlantVillage (treino) + metade PlantDoc (treino). Nada do teste.
i_rep_pv = np.sort(rng.choice(np.where(pv_split == 0)[0], N_REP // 2, replace=False))
i_rep_pd = rng.choice(np.where(pd_split == 0)[0], N_REP // 2, replace=False)
x_rep = np.concatenate([np.asarray(pv_x[i_rep_pv]), pd_x[i_rep_pd]]).astype(np.float32)
rng.shuffle(x_rep)
print(f'Avaliação: PV {len(y_te)} | PlantDoc {len(y_pd)} | calibração int8: {len(x_rep)}')
print('TF', tf.__version__, '| Keras', keras.__version__)

## 2. Reconstruir os modelos em float32
Os modelos foram treinados com precisão mista (float16). Para exportar, reconstruímos a mesma arquitetura
em float32 e copiamos os pesos, assim o TFLite não fica com conversões float16 no meio do grafo.

In [ ]:
keras.mixed_precision.set_global_policy('float32')

def conv_bn(x, f):
    x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def build_model(name):
    inp = keras.Input((IMG, IMG, 3), name='image')
    if name == 'cnn_base':
        x = layers.Rescaling(1 / 255.0)(inp)
        for f in (32, 64, 128, 256):
            x = conv_bn(x, f); x = conv_bn(x, f)
            x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.3)(x)
    elif name == 'mnv3s':
        base = keras.applications.MobileNetV3Small(input_shape=(IMG, IMG, 3), include_top=False, weights=None,
                                                   pooling='avg', include_preprocessing=True)
        x = layers.Dropout(0.2)(base(inp))
    elif name == 'mnv2_050':
        x = layers.Rescaling(1 / 127.5, offset=-1.0)(inp)
        base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), alpha=0.5, include_top=False,
                                              weights=None, pooling='avg')
        x = layers.Dropout(0.2)(base(x))
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='probs')(x)
    return keras.Model(inp, out, name=name)

models = {}
for r in RUNS:
    src = keras.models.load_model(f'{RUNS_DIR}/{r}_aug_ft/best.keras', compile=False)
    m = build_model(r)
    m.set_weights(src.get_weights())
    # conferência: as duas versões devem prever igual (diferença só de arredondamento do float16)
    a = src.predict(x_pd[:64].astype(np.float32), verbose=0)
    b = m.predict(x_pd[:64].astype(np.float32), verbose=0)
    print(f'{r:10s} concordância top-1 mixed×float32: {(a.argmax(1) == b.argmax(1)).mean():.3f} '
          f'| máx dif. prob: {np.abs(a - b).max():.4f}')
    models[r] = m
    del src

## 3. Conversão para TFLite

In [ ]:
def representative_dataset():
    for i in range(len(x_rep)):
        yield [x_rep[i:i + 1]]

def convert(model, run):
    sm_dir = f'/content/saved_model_{run}'
    model.export(sm_dir)
    paths = {}
    # float32
    conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    paths['fp32'] = f'{OUT_DIR}/{run}_fp32.tflite'
    open(paths['fp32'], 'wb').write(conv.convert())
    # int8 (pesos e ativações), entrada e saída continuam float32 para simplificar o app
    conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = representative_dataset
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.TFLITE_BUILTINS]
    paths['int8'] = f'{OUT_DIR}/{run}_int8.tflite'
    open(paths['int8'], 'wb').write(conv.convert())
    return paths

PATHS = {}
for r in RUNS:
    t0 = time.time()
    PATHS[r] = convert(models[r], r)
    print(f'{r:10s} ' + ' | '.join(f'{k}: {os.path.getsize(p)/1e6:.2f} MB' for k, p in PATHS[r].items())
          + f' ({time.time()-t0:.0f}s)')

## 4. Avaliação dos arquivos .tflite

In [ ]:
def tflite_predict(path, x, batch=32):
    it = tf.lite.Interpreter(model_path=path, num_threads=os.cpu_count())
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    it.resize_tensor_input(inp['index'], [batch, IMG, IMG, 3]); it.allocate_tensors()
    preds = []
    for i in range(0, len(x), batch):
        xb = x[i:i + batch].astype(np.float32); n = len(xb)
        if n < batch:
            xb = np.concatenate([xb, np.zeros((batch - n, IMG, IMG, 3), np.float32)])
        it.set_tensor(inp['index'], xb); it.invoke()
        preds.append(it.get_tensor(out['index'])[:n].astype(np.float32))
    return np.concatenate(preds)

def latency_ms(path, threads=4, n=50):
    it = tf.lite.Interpreter(model_path=path, num_threads=threads); it.allocate_tensors()
    inp = it.get_input_details()[0]
    xb = x_pd[:1].astype(np.float32)
    for _ in range(5):
        it.set_tensor(inp['index'], xb); it.invoke()
    t0 = time.perf_counter()
    for _ in range(n):
        it.set_tensor(inp['index'], xb); it.invoke()
    return 1000 * (time.perf_counter() - t0) / n

def metrics(p_pv, p_pd):
    pd_pred = p_pd.argmax(1); k = p_pd.max(1) >= THRESHOLD
    return {'pv_acc': accuracy_score(y_te, p_pv.argmax(1)),
            'pd_acc': accuracy_score(y_pd, pd_pred),
            'pd_f1': f1_score(y_pd, pd_pred, labels=PD_IDX, average='macro', zero_division=0),
            f'pd_cob@{THRESHOLD}': float(k.mean()),
            f'pd_acc@{THRESHOLD}': float((pd_pred[k] == y_pd[k]).mean()) if k.any() else float('nan')}

rows = []
for r in RUNS:
    m = models[r]
    rows.append({'modelo': NAMES[r], 'formato': 'keras', 'params': m.count_params(), 'tamanho_MB': np.nan,
                 'lat_ms_cpu': np.nan,
                 **metrics(m.predict(x_te.astype(np.float32), verbose=0), m.predict(x_pd.astype(np.float32), verbose=0))})
    for fmt, path in PATHS[r].items():
        t0 = time.time()
        row = {'modelo': NAMES[r], 'formato': fmt, 'params': m.count_params(),
               'tamanho_MB': os.path.getsize(path) / 1e6, 'lat_ms_cpu': latency_ms(path),
               **metrics(tflite_predict(path, x_te), tflite_predict(path, x_pd))}
        rows.append(row)
        print(f'{r:10s} {fmt:5s} PV {row["pv_acc"]:.3f} | PlantDoc {row["pd_acc"]:.3f} '
              f'| {row["tamanho_MB"]:.2f} MB | {row["lat_ms_cpu"]:.1f} ms ({time.time()-t0:.0f}s)')

tab = pd.DataFrame(rows)
tab.to_csv(f'{TAB_DIR}/tab_tflite.csv', index=False)
tex = tab.copy()
for c in ['pv_acc', 'pd_acc', 'pd_f1', f'pd_cob@{THRESHOLD}', f'pd_acc@{THRESHOLD}']:
    tex[c] = (100 * tex[c]).round(1)
tex['params'] = (tex['params'] / 1e6).round(2)
tex.to_latex(f'{TAB_DIR}/tab_tflite.tex', index=False, float_format='%.2f', na_rep='--')
tab.round(3)

## 5. Rótulos e metadados para o app

In [ ]:
with open(f'{OUT_DIR}/labels.txt', 'w') as f:
    f.write('\n'.join(PV_CLASSES) + '\n')

info = {
    'input': {'shape': [1, IMG, IMG, 3], 'dtype': 'float32', 'range': [0, 255], 'color': 'RGB',
              'preprocess': 'recorte quadrado central + redimensionamento bilinear para 128x128'},
    'output': {'shape': [1, NUM_CLASSES], 'dtype': 'float32', 'type': 'softmax'},
    'labels_file': 'labels.txt',
    'threshold': THRESHOLD,
    'pd_class_indices': PD_IDX.tolist(),
    'models': {r: {'name': NAMES[r], 'fp32': os.path.basename(PATHS[r]['fp32']),
                   'int8': os.path.basename(PATHS[r]['int8'])} for r in RUNS},
}
json.dump(info, open(f'{OUT_DIR}/model_info.json', 'w'), indent=2, ensure_ascii=False)
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{f:28s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.2f} MB')

## 6. Resumo para colar no chat
Depois de rodar, avise: os arquivos `.tflite`, `labels.txt` e `model_info.json` serão copiados para
a pasta `models/` do repositório e para o app.

In [ ]:
print(tab.to_string(index=False, float_format='%.3f'))